# TuniShop — Machine learning 07. Publication de tous les résultats dans TuniShop_ANALYTICS

**Auteure :** Sarra Abdeljaouad · **Phase :** machine learning · **Source :** TuniShop_DWH (vues `bi`), via `05_Data/features` · **Découpage :** apprentissage 2023-2024, test 2025

## Objectif

Chaque notebook de l'EDA, du data mining et du ML écrit déjà ses résultats à **deux endroits** : le fichier (CSV ou Parquet, comme avant) **et** une table du schéma `ana` de la base SQL Server **TuniShop_ANALYTICS**. Ce notebook sert à :

1. **tout republier d'un coup** à partir des fichiers déjà produits (utile la première fois, ou si la base a été recréée) ;
2. **contrôler** que la base contient bien les mêmes lignes que les fichiers, et que les clés retrouvent leur dimension dans le DWH ;
3. afficher le **journal** des publications (`ctl.journal_publication`).

**Prérequis (une seule fois)** : exécuter dans SSMS les scripts `00_Commun/sql/01` à `04` (base, tables, vues pour Power BI, contrôles).

**Quand l'exécuter ?** En dernier, après `05_Evaluation_Modeles`. Il ne lit pas le DWH : il relit les fichiers de `06_Exports` et `05_Data`.

## 0. Préparation

In [1]:
# --- Bibliothèques
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# --- Module commun du projet (dossier 00_Commun)
sys.path.insert(0, "../00_Commun")
from tunishop_analytics import config as cfg, chargement, features, temporal_split, validation, publication

warnings.filterwarnings("ignore")
cfg.appliquer_style()
sql = chargement.requete                       # sql("SELECT ...") -> exécuté par SQL Server
pd.set_option("display.float_format", lambda v: f"{v:,.3f}".replace(",", " "))

def montrer(nom):
    """Enregistre la figure en cours dans 06_Exports/Figures/<nom>.png, puis l'affiche."""
    cfg.sauver_figure(plt.gcf(), nom)
    plt.show()

for dossier in (cfg.DOSSIER_MODELES, cfg.DOSSIER_PREDICTIONS):
    dossier.mkdir(parents=True, exist_ok=True)

In [2]:
# Où écrit-on ? (base de test SQLite ou TuniShop_ANALYTICS sur le serveur de config.py)
print("Publication activée :", publication.actif())
publication.ouvrir_connexion();

Publication activée : True
Publication : connecté à TuniShop_ANALYTICS sur localhost


## 1. Republier tous les fichiers de résultats

Chaque fichier donne une table (règle de nommage dans `publication.nom_table`) :

| Fichier | Table dans TuniShop_ANALYTICS |
|---|---|
| `06_Exports/Tables/Q01_ventes_annuelles.csv` | `ana.eda_q01_ventes_annuelles` |
| `06_Exports/Tables/EDA_hypotheses_02_EDA_Sales.csv` (et les 4 autres) | `ana.eda_hypotheses` (une ligne par question) |
| `06_Exports/Tables/DM01_profil_segments.csv` | `ana.dm01_profil_segments` |
| `06_Exports/Tables/ML_metriques_churn.csv` | `ana.ml_metriques_churn` |
| `05_Data/features/segments_rfm.parquet` | `ana.dm_segments_rfm` |
| `06_Exports/Predictions/churn_2025_test.parquet` | `ana.ml_pred_churn_2025` |

Chaque table est **vidée puis remplie** en une seule transaction : en cas d'erreur, l'ancien contenu reste en place.

In [3]:
bilan = publication.republier_tout()
print(f"{(bilan['statut'] == 'OK').sum()} tables publiées sur {len(bilan)} ;",
      f"{int(bilan['lignes'].sum()):,} lignes au total".replace(",", " "))
bilan[bilan["statut"] != "OK"] if (bilan["statut"] != "OK").any() else bilan.head(10)

  ✔ ana.dm01_actions_segments                         8 lignes
  ✔ ana.dm01_churn_par_segment                        8 lignes
  ✔ ana.dm01_migrations                               8 lignes
  ✔ ana.dm01_profil_segments                          8 lignes
  ✔ ana.dm02_choix_k                                  7 lignes
  ✔ ana.dm02_profil_clusters                          4 lignes
  ✔ ana.dm02_stabilite_clusters                       4 lignes
  ✔ ana.dm03_evaluation_recommandation                3 lignes
  ✔ ana.dm03_lift_paires_categories                   9 lignes
  ✔ ana.dm03_regles_retenues                         86 lignes
  ✔ ana.dm03_taille_paniers                           5 lignes
  ✔ ana.dm04_composition_paniers                      4 lignes
  ✔ ana.dm04_coupon_fidelite                          2 lignes
  ✔ ana.dm04_facteurs_rachat                         19 lignes
  ✔ ana.eda00_coherence                               5 lignes
  ✔ ana.eda00_extremes                                7

,fichier,table,lignes,statut
0,DM01_actions_segments.csv,ana.dm01_actions_segments,8,OK
1,DM01_churn_par_segment.csv,ana.dm01_churn_par_segment,8,OK
2,DM01_migrations.csv,ana.dm01_migrations,8,OK
3,DM01_profil_segments.csv,ana.dm01_profil_segments,8,OK
4,DM02_choix_k.csv,ana.dm02_choix_k,7,OK
5,DM02_profil_clusters.csv,ana.dm02_profil_clusters,4,OK
6,DM02_stabilite_clusters.csv,ana.dm02_stabilite_clusters,4,OK
7,DM03_evaluation_recommandation.csv,ana.dm03_evaluation_recommandation,3,OK
8,DM03_lift_paires_categories.csv,ana.dm03_lift_paires_categories,9,OK
9,DM03_regles_retenues.csv,ana.dm03_regles_retenues,86,OK


## 2. Contrôles

**2.1 Mêmes lignes dans la base et dans les fichiers.** On recompte chaque table dans la base et on compare au fichier relu.

In [4]:
cn = publication.ouvrir_connexion()
cur = cn.cursor()
def lignes_en_base(table):
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    return cur.fetchone()[0]

# Une table partagée (eda_hypotheses) reçoit plusieurs fichiers : on compare la somme
attendu = bilan.groupby("table")["lignes"].sum()
controle = pd.DataFrame({"lignes_fichiers": attendu,
                         "lignes_base": [lignes_en_base(t) for t in attendu.index]})
controle["statut"] = np.where(controle["lignes_fichiers"] == controle["lignes_base"], "OK", "ÉCART")
print(f"{(controle['statut'] == 'OK').sum()} / {len(controle)} tables identiques aux fichiers")
controle[controle["statut"] != "OK"] if (controle["statut"] != "OK").any() else controle.head(8)

88 / 88 tables identiques aux fichiers


,lignes_fichiers,lignes_base,statut
table,,,
ana.dm01_actions_segments,8,8,OK
ana.dm01_churn_par_segment,8,8,OK
ana.dm01_migrations,8,8,OK
ana.dm01_profil_segments,8,8,OK
ana.dm02_choix_k,7,7,OK
ana.dm02_profil_clusters,4,4,OK
ana.dm02_stabilite_clusters,4,4,OK
ana.dm03_evaluation_recommandation,3,3,OK


**2.2 Liens avec le DWH.** Une clé étrangère ne peut pas pointer vers une autre base : le lien est donc vérifié par la procédure `ctl.usp_controler_publication` (script `00_Commun/sql/04_Controles_TuniShop_ANALYTICS.sql`). Elle compte, pour chaque table de résultats et de scores (L1-L7), les clés absentes du DWH (`client_sk`, `produit_id`, `livraison_id`) : on attend **0 partout**, comme le contrôle C08 de l'EDA.

En mode test (SQLite), la procédure n'existe pas : le contrôle est fait en Python sur le miroir des vues `bi`.

In [5]:
if publication._sqlite:
    # Mode test : même contrôle en Python
    clients = set(sql("SELECT client_sk FROM bi.v_dim_client")["client_sk"])
    produits = set(sql("SELECT produit_id FROM bi.v_dim_produit")["produit_id"])
    livraisons = set(sql("SELECT livraison_id FROM bi.v_fact_livraisons")["livraison_id"])
    verif = []
    for table, col, ref in [("dm_segments_rfm", "client_sk", clients), ("dm_clusters_clients", "client_sk", clients),
                            ("ml_pred_churn_2025", "client_sk", clients), ("ml_pred_demande_2025", "produit_id", produits),
                            ("ml_pred_retard_2025", "livraison_id", livraisons), ("ml_scores_churn_actuels", "client_sk", clients),
                            ("ml_scores_retard_en_cours", "livraison_id", livraisons)]:
        valeurs = pd.read_sql(f"SELECT DISTINCT [{col}] FROM ana.{table}", cn)[col]
        orphelins = int((~valeurs.isin(ref)).sum())
        verif.append({"table": "ana." + table, "cle": col, "orphelins": orphelins, "statut": "OK" if orphelins == 0 else "ÉCART"})
    liens = pd.DataFrame(verif)
else:
    liens = pd.read_sql("EXEC ctl.usp_controler_publication", cn)
liens

,controle,table_sql,verification,attendu,obtenu,statut
0,H1,ana.ml_suivi_historique,lignes de la dernière exécution = lignes de an...,171,171,OK
1,L1,ana.dm_segments_rfm,client_sk absent de bi.v_dim_client,0,0,OK
2,L2,ana.dm_clusters_clients,client_sk absent de bi.v_dim_client,0,0,OK
3,L3,ana.ml_pred_churn_2025,client_sk absent de bi.v_dim_client,0,0,OK
4,L4,ana.ml_pred_demande_2025,produit_id absent de bi.v_dim_produit (version...,0,0,OK
...,...,...,...,...,...,...
180,P1,ana.ml08_retard_en_cours,publiée au moins une fois,1,1,OK
181,P1,ana.tm_avis,publiée au moins une fois,1,1,OK
182,P1,ana.tm_avis_themes,publiée au moins une fois,1,1,OK
183,P1,ana.tm_produits,publiée au moins une fois,1,1,OK


## 3. Journal des publications

In [6]:
publication.journal(15)

,id_publication,table_sql,source_fichier,partage,lignes,colonnes,mode_source,publie_le,publie_par
0,5077,ana.dm_regles_association,06_Exports/Modeles/regles_association.parquet,None,260,10,DWH,2026-11-20 12:20:20,Abdeljaouad
1,5076,ana.ml_pred_ventes_2026,06_Exports/Predictions/ventes_2026.parquet,None,108,3,DWH,2026-11-20 12:20:20,Abdeljaouad
2,5075,ana.ml_pred_ventes_2025,06_Exports/Predictions/ventes_2025_test.parquet,None,108,7,DWH,2026-11-20 12:20:20,Abdeljaouad
3,5074,ana.ml_scores_retard_en_cours,06_Exports/Predictions/retard_scores_en_cours....,None,211,11,DWH,2026-11-20 12:20:20,Abdeljaouad
4,5073,ana.ml_pred_retard_2025,06_Exports/Predictions/retard_2025_test.parquet,None,26590,5,DWH,2026-11-20 12:20:20,Abdeljaouad
5,5072,ana.ml_scores_demande,06_Exports/Predictions/demande_scores_annee_su...,None,7140,9,DWH,2026-11-20 12:20:19,Abdeljaouad
6,5071,ana.ml_pred_demande_2025,06_Exports/Predictions/demande_2025_test.parquet,None,6992,10,DWH,2026-11-20 12:20:18,Abdeljaouad
7,5070,ana.ml_scores_churn_actuels,06_Exports/Predictions/churn_scores_actuels.pa...,None,7377,11,DWH,2026-11-20 12:20:18,Abdeljaouad
8,5069,ana.ml_pred_churn_2025,06_Exports/Predictions/churn_2025_test.parquet,None,19414,5,DWH,2026-11-20 12:20:17,Abdeljaouad
9,5068,ana.dm_segments_rfm,05_Data/features/segments_rfm.parquet,None,43488,8,DWH,2026-11-20 12:20:17,Abdeljaouad


## 4. Lignage et dictionnaire jusqu'à Power BI

`05_Data/metadata/Data_Lineage.xlsx` et `Dictionnaire_Variables.xlsx` reçoivent (ou remplacent) les feuilles qui décrivent la fin de la chaîne : fichier de résultats → table `ana` → vue `pbi` → visuels Power BI, colonnes des tables et des vues, mesures DAX. Elles sont relues dans les objets réels du projet (`Schema_Base_Analytics.json`, script `03_Vues_pbi_Power_BI.sql`, projet `07_PowerBI`).

In [7]:
from tunishop_analytics import documentation
documentation.documenter()

Classeur enregistré : 05_Data\metadata\_tmp_Data_Lineage.xlsx
Classeur enregistré : 05_Data\metadata\_tmp_Dictionnaire_Variables.xlsx
Lignage et dictionnaire mis à jour : 92 fichiers -> tables, 20 vues, 583 colonnes ana, 0 colonnes Power BI, 0 mesures


In [8]:
cur.close()
publication.fermer_connexion()
chargement.fermer_connexion()
print("Publication terminée : Power BI peut être actualisé (vues pbi.* de TuniShop_ANALYTICS).")

Publication terminée : Power BI peut être actualisé (vues pbi.* de TuniShop_ANALYTICS).
